## Environment check

This cell verifies that the local `nbloss` package is imported from the intended project environment before running the benchmark. The package provides the Smooth Net Benefit training procedure and Net Benefit evaluation functions used later in the GAM experiments.

In [12]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


## Step 1 — TabZilla benchmark datasets

This step defines the OpenML datasets included in the TabZilla GAM benchmark. The dataset identifiers are fixed in advance so that the same collection of binary classification tasks can be evaluated across training objectives and model classes.

Datasets are subsequently retrieved directly from OpenML. Eligibility checks, including binary-outcome verification and minimum sample-size requirements, are applied during benchmark execution.

In [13]:
OPENML_DATASET_IDS = [
    1120,   # Magic telescope
    1053,   # openml__jm1__3904
    4532,   # higgs
    4534,   # phishing websites
    1489,   # phoneme
    1502,   # skin segmentation
    1590,   # adult income
    45072,  # airlines
    151,    # electricity
    4135,   # Amazon_employee_access
    40978,  # internet advertisements
    41434,  # click prediction small
    41150,  # miniBooNE
    40536,  # speeddating
    1043,   # ada agnostic
    1462,   # banknote authentication
    41142,  # christine
    40701,  # churn
    31,     # credit-g
    1471,   # eeg-state
    846,    # elevators
    1038,   # gina agnostic
    821,    # house 16H
    41143,  # jasmine
    1067,   # kc1
    1485,   # madelon
    24,     # mushroom
    1116,   # musk
    1486,   # nomao
    23517,  # numerai28.6
    1487,   # ozone
    1068,   # pc1
    1050,   # pc3
    1049,   # pc4
    41145,  # philippine
    871,    # pollen
    312,    # scene
    38,     # sick
    44,     # spambase
    1570,   # wilt
    45035,  # Albert
    1461,   # bank marketing
    1036,   # sylvia agnostic
    41146,  # sylvine
]

## Step 2 — Cross-validation splits

This step defines the outer train/test partitions used throughout the benchmark. Each dataset is split using stratified 5-fold cross-validation with a fixed random seed.

For each of the five runs, one fold is used as the independent test set and the remaining four folds form the training set. This rotating-fold design ensures that every observation is used for testing exactly once.

The same fold construction is used across training objectives and can be shared across model classes, allowing comparisons to be performed on identical participant-level or observation-level test sets.

In [14]:

from sklearn.model_selection import StratifiedKFold

def make_5fold_indices(y: np.ndarray, seed: int):
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )
    return [test_idx for _, test_idx in skf.split(np.zeros_like(y), y)]


def indices_for_run_train_test(folds, run_id: int):
    """
    5-fold rotation:
    - run_id fold = test
    - all other folds = train
    """
    test_fold = run_id % 5

    test_inds = folds[test_fold]

    train_inds = np.concatenate(
        [folds[i] for i in range(5) if i != test_fold],
        axis=0,
    )

    return train_inds, test_inds

## Step 3 — GAM preprocessing

This step constructs the design matrices used by the generalized additive models.

Feature types are inferred from the outer training fold. Binary variables are represented as single features, string variables and numeric variables with at most 20 unique values are treated as categorical, and remaining numeric variables are treated as continuous.

Categorical variables are restricted to their 10 most frequent levels, with less frequent values grouped into an `OTHER` category, and are subsequently one-hot encoded. Continuous variables are median-imputed, standardized, and expanded using cubic spline basis functions. Binary variables are retained without spline expansion.

All preprocessing transformations are fitted on the outer training fold only and then applied unchanged to the corresponding test fold.

The preprocessing step additionally records the locations of individual spline blocks and the indices of non-spline coefficients. These metadata are used later to apply separate regularization penalties to linear/categorical terms and spline terms.

In [15]:

import numpy as np
import pandas as pd

from dataclasses import dataclass
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def _to_dense_float32(X):
    if sparse.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=np.float32)


def _make_ohe():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop="if_binary",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop="if_binary",
            sparse=False,
        )


def _make_spline_transformer(*, n_knots: int = 10, degree: int = 3):
    kwargs = dict(
        n_knots=int(n_knots),
        degree=int(degree),
        include_bias=False,
    )
    try:
        return SplineTransformer(**kwargs, sparse_output=False)
    except TypeError:
        return SplineTransformer(**kwargs, sparse=False)


# ----------------------------------------------------------------------
# Feature inference
# ----------------------------------------------------------------------
def infer_feature_types(X: pd.DataFrame, *, numeric_cat_max_unique: int = 20):
    binary_cols = []
    categorical_cols = []
    numeric_cols = []

    for c in X.columns:
        s = X[c]
        nunq_including_nan = pd.Series(s).nunique(dropna=False)

        if nunq_including_nan == 2:
            binary_cols.append(c)
            continue

        dtype_name = str(s.dtype)

        if dtype_name in ("object", "category", "string"):
            categorical_cols.append(c)
            continue

        if pd.api.types.is_numeric_dtype(s):
            if pd.Series(s).nunique(dropna=False) <= numeric_cat_max_unique:
                categorical_cols.append(c)
            else:
                numeric_cols.append(c)
            continue

        categorical_cols.append(c)

    return binary_cols, categorical_cols, numeric_cols


# ----------------------------------------------------------------------
# Top-K grouping for categoricals
# ----------------------------------------------------------------------
class TopKCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, top_k: int = 10):
        self.top_k = int(top_k)
        self.keep_values_ = None
        self.columns_ = None

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)
        self.keep_values_ = {}

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_values_[c] = set(vc.head(self.top_k).index.tolist())

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = pd.DataFrame(index=X_df.index)

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            keep = self.keep_values_[c]
            out[c] = s.where(s.isin(keep), "__OTHER__")

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


# ----------------------------------------------------------------------
# Binary single-column encoder
# ----------------------------------------------------------------------
class BinaryPassthroughEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns_ = None
        self.fill_values_ = {}
        self.value_maps_ = {}

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)

        for c in self.columns_:
            s = X_df[c]
            non_missing = s[~pd.isna(s)]

            if len(non_missing) == 0:
                self.fill_values_[c] = 0
                self.value_maps_[c] = {}
                continue

            mode_vals = non_missing.mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else non_missing.iloc[0]
            self.fill_values_[c] = fill_val

            seen = []
            for v in non_missing:
                if v not in seen:
                    seen.append(v)

            if len(seen) == 1:
                mapping = {seen[0]: 0.0}
            else:
                mapping = {seen[0]: 0.0, seen[1]: 1.0}

            self.value_maps_[c] = mapping

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = np.zeros((len(X_df), len(self.columns_)), dtype=np.float32)

        for j, c in enumerate(self.columns_):
            s = X_df[c].copy()
            fill_val = self.fill_values_[c]
            mapping = self.value_maps_[c]

            s = s.where(~pd.isna(s), fill_val)
            default_code = mapping.get(fill_val, 0.0)

            out[:, j] = s.map(lambda v: mapping.get(v, default_code)).astype(np.float32).to_numpy()

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


# ----------------------------------------------------------------------
# Metadata bundle
# ----------------------------------------------------------------------
@dataclass
class GamPreprocessorBundle:
    binary_cols: list
    categorical_cols: list
    numeric_cols: list

    preprocessor_gam: ColumnTransformer
    gam_feature_names: list

    ohe_dim: int
    spline_dim: int
    bin_dim: int

    spline_blocks: dict
    spline_block_slices: list
    non_spline_idx: np.ndarray


# ----------------------------------------------------------------------
# Fit GAM preprocessor on TRAIN only
# ----------------------------------------------------------------------
def _fit_gam_preprocessor_bundle(
    X_tr: pd.DataFrame,
    *,
    numeric_cat_max_unique: int = 20,
    top_k_categories: int = 10,
    spline_n_knots: int = 10,
    spline_degree: int = 3,
):
    binary_cols, categorical_cols, numeric_cols = infer_feature_types(
        X_tr,
        numeric_cat_max_unique=numeric_cat_max_unique,
    )

    cat_pipe = Pipeline(
        steps=[
            ("topk", TopKCategoryGrouper(top_k=top_k_categories)),
            ("ohe", _make_ohe()),
        ]
    )

    num_pipe_gam = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("spline", _make_spline_transformer(
                n_knots=spline_n_knots,
                degree=spline_degree,
            )),
        ]
    )

    bin_pipe = Pipeline(
        steps=[
            ("binary", BinaryPassthroughEncoder()),
        ]
    )

    preprocessor_gam = ColumnTransformer(
        transformers=[
            ("cat", cat_pipe, categorical_cols),
            ("num", num_pipe_gam, numeric_cols),
            ("bin", bin_pipe, binary_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

    preprocessor_gam.fit(X_tr)

    try:
        gam_feature_names = list(preprocessor_gam.get_feature_names_out())
    except Exception:
        gam_feature_names = [
            f"phi{i}"
            for i in range(preprocessor_gam.transform(X_tr.iloc[:1]).shape[1])
        ]

    # OHE dimension
    if len(categorical_cols) > 0:
        X_cat_tr = preprocessor_gam.named_transformers_["cat"].transform(X_tr[categorical_cols])
        ohe_dim = int(_to_dense_float32(X_cat_tr).shape[1])
    else:
        ohe_dim = 0

    bin_dim = int(len(binary_cols))

    spline_blocks = {}
    spline_block_slices = []
    spline_dim = 0

    if len(numeric_cols) > 0:
        spline = preprocessor_gam.named_transformers_["num"].named_steps["spline"]
        spline_dim = int(spline.n_features_out_)
        per_feature_dim = spline_dim // len(numeric_cols)

        start = ohe_dim
        for col in numeric_cols:
            end = start + per_feature_dim
            idxs = list(range(start, end))
            spline_blocks[col] = idxs
            spline_block_slices.append((col, start, end))
            start = end

    non_spline_idx = np.array(
        list(range(ohe_dim)) +
        list(range(ohe_dim + spline_dim, ohe_dim + spline_dim + bin_dim)),
        dtype=np.int64,
    )

    return GamPreprocessorBundle(
        binary_cols=binary_cols,
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        preprocessor_gam=preprocessor_gam,
        gam_feature_names=gam_feature_names,
        ohe_dim=ohe_dim,
        spline_dim=spline_dim,
        bin_dim=bin_dim,
        spline_blocks=spline_blocks,
        spline_block_slices=spline_block_slices,
        non_spline_idx=non_spline_idx,
    )


# ----------------------------------------------------------------------
# Public API: GAM matrix + spline metadata
# ----------------------------------------------------------------------
def fit_train_preprocessor_spline_and_transform(
    X_tr: pd.DataFrame,
    X_te: pd.DataFrame,
    *,
    spline_n_knots: int = 10,
    spline_degree: int = 3,
):
    """
    New GAM range experiment:
    - Fit on TRAIN only
    - Transform TRAIN + TEST

    Returns dict with:
      Phi_tr, Phi_te, spline metadata, and feature groups.
    """
    bundle = _fit_gam_preprocessor_bundle(
        X_tr,
        spline_n_knots=spline_n_knots,
        spline_degree=spline_degree,
    )

    Phi_tr = _to_dense_float32(bundle.preprocessor_gam.transform(X_tr))
    Phi_te = _to_dense_float32(bundle.preprocessor_gam.transform(X_te))

    if np.isnan(Phi_tr).any() or np.isnan(Phi_te).any():
        raise RuntimeError("GAM preprocessing produced NaNs.")

    spline = None
    if len(bundle.numeric_cols) > 0:
        spline = bundle.preprocessor_gam.named_transformers_["num"].named_steps["spline"]

    return {
        "bundle": bundle,
        "enc": bundle,
        "spline": spline,

        "Phi_tr": Phi_tr,
        "Phi_te": Phi_te,

        "binary_cols": bundle.binary_cols,
        "categorical_cols": bundle.categorical_cols,
        "numeric_cols": bundle.numeric_cols,

        "spline_blocks": bundle.spline_blocks,
        "spline_block_slices": bundle.spline_block_slices,
        "non_spline_idx": bundle.non_spline_idx,

        "gam_feature_names": bundle.gam_feature_names,

        "ohe_dim": bundle.ohe_dim,
        "spline_dim": bundle.spline_dim,
        "bin_dim": bundle.bin_dim,
    }

## Step 4 — Global configuration and GAM model

This step defines the global settings used throughout the GAM benchmark.

It specifies the computational device, random seeds, batch size, minimum dataset size, Smooth Net Benefit annealing schedule, Net Benefit threshold-grid resolution, local calibration settings, and the regularization grids used for GAM training.

Smooth Net Benefit training uses inverse-temperature annealing with values 1, 4, and 10. Decision performance is evaluated over a threshold band of ±0.025 around each reference threshold.

Two regularization parameters are considered for the GAM: a ridge-type penalty for non-spline coefficients and a smoothness penalty for the spline coefficients. Their candidate values are defined here for subsequent inner cross-validation.

In [23]:
import random

import numpy as np
import torch
import torch.nn as nn

from nbloss.trainer import set_seed


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

SEED_GLOBAL = 1234
MODEL_SEED = 4242

BATCH = 1024
MIN_ROWS = 1000

set_seed(SEED_GLOBAL)

LR_ADAMW = 3e-2
INVERSE_TEMPS = (1.0, 4.0, 10.0)
EPOCHS_PER_TEMP = 300
PATIENCE_HARD = 20
TRAIN_RANGE_POINTS = 11
TEST_RANGE_POINTS = 201

BAND_HALF_WIDTH = 0.025
MID_PREV_LOW, MID_PREV_HIGH = 0.40, 0.60

LAMBDA_LIN_GRID = (0.0, 1e-4, 1e-3, 1e-2)
LAMBDA_SMOOTH_GRID = (1e-4, 1e-3, 1e-2, 1e-1)

LOCAL_START_HALF_WIDTH = 0.1
LOCAL_EXPAND_STEP = 0.01
LOCAL_MIN_POS = 50
LOCAL_MIN_NEG = 50

TEMP_MAX_ITER = 500
PLATT_MAX_ITER = 500


def band_from_t_ref(t_ref: float, half_width: float = BAND_HALF_WIDTH) -> tuple[float, float]:
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)


class TorchGAM(nn.Module):
    def __init__(self, d_in_phi: int):
        super().__init__()
        self.linear = nn.Linear(d_in_phi, 1, bias=True)

    def forward(self, phi_x):
        return self.linear(phi_x).squeeze(-1)


def optimizer_factory_no_wd(model: torch.nn.Module):

    return make_optimizer(
        model,
        name="adamw",
        lr=LR_ADAMW,
        weight_decay=0.0,
    )


## Step 5 — OpenML loading, output utilities, and threshold specification

This step defines utility functions for loading benchmark datasets from OpenML and managing benchmark outputs.

For each OpenML dataset, the predictors and binary target are retrieved together with dataset metadata such as sample size, number of raw predictors, prevalence, and class labels.

Reference decision thresholds are derived from the prevalence of the outer training fold. The prevalence threshold is evaluated for every dataset. For datasets with prevalence outside the central 40%–60% range, an additional inverse-prevalence threshold is evaluated.

For each reference threshold, Net Benefit is evaluated across a decision-relevant interval of ±0.025 around the reference threshold.

The block also contains helper functions for reproducible PyTorch data loaders and safe incremental storage of benchmark results.

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import openml


# ============================================================================
# Output helpers
# ============================================================================

def _append_csv_safely(df: pd.DataFrame, path: Path):
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def _read_done_registry(path_done: Path) -> set[int]:
    if not path_done.exists():
        return set()

    try:
        df = pd.read_csv(path_done)
        return set(df["openml_id"].astype(int).tolist())
    except Exception:
        return set()


def _mark_dataset_done(openml_id: int, name: str, path_done: Path):
    _append_csv_safely(
        pd.DataFrame([{"openml_id": int(openml_id), "name": str(name)}]),
        path_done,
    )


def make_loader(X, y, batch=BATCH, shuffle=False, seed=SEED_GLOBAL):
    g = torch.Generator().manual_seed(seed)

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

    return DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        generator=g,
    )

# ============================================================================
# OpenML loader
# ============================================================================

def load_openml_dataset(did: int):
    ds = openml.datasets.get_dataset(did)

    X_df, y_raw, _, _ = ds.get_data(
        target=ds.default_target_attribute,
        dataset_format="dataframe",
        include_row_id=False,
        include_ignore_attribute=False,
    )

    y_series = pd.Series(y_raw)

    if (
        y_series.dtype.kind in ("U", "S", "O", "b")
        or str(y_series.dtype).startswith("category")
    ):
        y_cat = y_series.astype("category")
        classes = list(y_cat.cat.categories)
        y = y_cat.cat.codes.astype("int64").to_numpy().astype("float32")
    else:
        y = y_series.astype("int64").to_numpy().astype("float32")
        classes = ["0", "1"]

    prev = float((y == 1).mean())

    meta = dict(
        openml_id=int(ds.dataset_id),
        name=str(ds.name),
        n_rows=int(len(y)),
        n_features=int(X_df.shape[1]),
        prevalence=prev,
        pos_label=(classes[1] if len(classes) == 2 else "1"),
        neg_label=(classes[0] if len(classes) == 2 else "0"),
    )

    return ds, X_df, y, meta


# ============================================================================
# Threshold helpers
# ============================================================================

def threshold_specs_from_prevalence(
    prev: float,
    *,
    mid_prev_low: float = 0.40,
    mid_prev_high: float = 0.60,
):
    specs = [
        {
            "threshold_name": "prev",
            "threshold": float(prev),
        }
    ]

    if prev < mid_prev_low or prev > mid_prev_high:
        specs.append(
            {
                "threshold_name": "inverse",
                "threshold": float(1.0 - prev),
            }
        )

    return specs


def band_from_t_ref(
    t_ref: float,
    *,
    half_width: float = 0.025,
):
    eps = 1e-9
    t_min = max(0.0 + eps, float(t_ref) - float(half_width))
    t_max = min(1.0 - eps, float(t_ref) + float(half_width))

    if not (0.0 < t_min < t_max < 1.0):
        mid = min(max(float(t_ref), eps), 1.0 - eps)
        span = min(mid - eps, 1.0 - eps - mid, float(half_width))
        t_min, t_max = mid - span, mid + span

    return float(t_min), float(t_max)

## Step 6 — Shared GAM training utilities

This step defines the PyTorch data-loading and loss-evaluation utilities used during GAM training.

The design matrices produced by the preprocessing pipeline are converted to PyTorch tensors and supplied to the models through reproducible data loaders.

Helper functions are also defined for evaluating binary cross-entropy directly from model logits and for computing L2 penalties while excluding intercept terms. These utilities are shared by the baseline and decision-focused training procedures.

In [18]:
from torch.utils.data import DataLoader, TensorDataset


def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    *,
    batch: int = 1024,
    shuffle: bool = False,
    seed: int = 1234,
) -> DataLoader:
    g = torch.Generator().manual_seed(int(seed))

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).view(-1),
    )

    return DataLoader(
        ds,
        batch_size=int(batch),
        shuffle=bool(shuffle),
        generator=g,
        drop_last=False,
    )


def l2_penalty_weights_only(model: nn.Module) -> torch.Tensor:
    device = next(model.parameters()).device
    l2 = torch.zeros((), device=device)

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.endswith("bias") or name in ("bias", "b0", "intercept"):
            continue
        l2 = l2 + (p * p).sum()

    return l2


@torch.no_grad()
def bce_logits_loss(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
) -> float:
    model.eval()
    logits = model(X).view(-1)
    y = y.view(-1)
    loss = nn.BCEWithLogitsLoss(reduction="mean")(logits, y)
    return float(loss.detach().cpu().item())

## Step 7 — BCE-trained GAM with inner cross-validation

This step defines the baseline GAM training procedure.

The GAM is trained by minimizing binary cross-entropy using the spline-expanded design matrix. Two separate regularization components are applied:

- a ridge penalty on non-spline coefficients, including categorical and binary terms;
- a second-order finite-difference smoothness penalty within each continuous-variable spline block.

The strengths of these penalties are selected by stratified inner cross-validation using only the outer training fold.

For each candidate pair of regularization parameters, the model is fitted on the inner training data and evaluated using binary cross-entropy on the corresponding validation fold. The combination with the lowest mean validation loss is selected and subsequently used to fit the BCE baseline on the full outer training set.

In [19]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold


def _get_weight_vector(model: nn.Module) -> torch.Tensor:
    """
    Return the 1D coefficient vector excluding intercept.
    Assumes model.linear is nn.Linear(d_in, 1, bias=True).
    """
    if not hasattr(model, "linear"):
        raise AttributeError("Expected model to have attribute 'linear'.")
    return model.linear.weight.view(-1)


def _linear_l2_penalty_from_idx(model: nn.Module, non_spline_idx) -> torch.Tensor:
    """
    Ridge penalty on NON-spline coefficients only.
    """
    w = _get_weight_vector(model)
    device = w.device

    if non_spline_idx is None or len(non_spline_idx) == 0:
        return torch.zeros((), device=device)

    idx = torch.as_tensor(non_spline_idx, dtype=torch.long, device=device)
    return (w[idx] ** 2).sum()


def _smoothness_penalty_from_blocks(model: nn.Module, spline_block_slices) -> torch.Tensor:
    """
    Second-order finite-difference penalty within each spline block.
    """
    w = _get_weight_vector(model)
    device = w.device
    pen = torch.zeros((), device=device)

    if spline_block_slices is None:
        return pen

    for _, start, end in spline_block_slices:
        beta = w[start:end]
        if beta.numel() < 3:
            continue
        d2 = beta[2:] - 2.0 * beta[1:-1] + beta[:-2]
        pen = pen + (d2 ** 2).sum()

    return pen


@torch.no_grad()
def _bce_logits_loss(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    logits = model(X).view(-1)
    y = y.view(-1)
    loss = nn.BCEWithLogitsLoss(reduction="mean")(logits, y)
    return float(loss.detach().cpu().item())


def fit_bce_gam_lbfgs(
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    *,
    lambda_lin: float,
    lambda_smooth: float,
    non_spline_idx,
    spline_block_slices,
    max_iter: int = 500,
    tol_grad: float = 1e-7,
    tol_change: float = 1e-9,
    history_size: int = 100,
    device: str = "cpu",
):
    """
    Convex BCE fit with LBFGS on a fixed GAM design matrix Phi.

    Objective:
        BCE
      + lambda_lin    * L2(non-spline coefficients)
      + lambda_smooth * smoothness_penalty(spline blocks)
    """
    device_t = torch.device(device)

    X = torch.tensor(X_tr, dtype=torch.float32, device=device_t)
    y = torch.tensor(y_tr, dtype=torch.float32, device=device_t).view(-1)

    model = make_model_fn().to(device_t)
    bce = nn.BCEWithLogitsLoss(reduction="mean")

    opt = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=int(max_iter),
        tolerance_grad=float(tol_grad),
        tolerance_change=float(tol_change),
        history_size=int(history_size),
        line_search_fn="strong_wolfe",
    )

    lambda_lin_t = torch.tensor(float(lambda_lin), device=device_t)
    lambda_smooth_t = torch.tensor(float(lambda_smooth), device=device_t)

    def closure():
        opt.zero_grad(set_to_none=True)

        logits = model(X).view(-1)
        loss_bce = bce(logits, y)

        pen_lin = _linear_l2_penalty_from_idx(model, non_spline_idx)
        pen_smooth = _smoothness_penalty_from_blocks(model, spline_block_slices)

        loss = loss_bce + lambda_lin_t * pen_lin + lambda_smooth_t * pen_smooth

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite LBFGS loss: {loss.detach().item()}")

        loss.backward()
        return loss

    opt.step(closure)

    train_bce = _bce_logits_loss(model, X, y)
    return model, train_bce


def select_gam_penalties_by_cv_bce(
    lambda_lin_grid,
    lambda_smooth_grid,
    *,
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    non_spline_idx,
    spline_block_slices,
    device: str = "cpu",
    n_splits: int = 5,
    lbfgs_max_iter: int = 500,
    seed: int = 1234,
):
    """
    Select (lambda_lin, lambda_smooth) using INNER CV on training data.

    Returns:
        lambda_lin_star, lambda_smooth_star, best_cv_bce, table_rows
    """
    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    device_t = torch.device(device)

    best = None
    rows = []

    for lambda_lin in lambda_lin_grid:
        for lambda_smooth in lambda_smooth_grid:
            fold_bces = []

            for fold_id, (inner_tr_idx, inner_va_idx) in enumerate(skf.split(X_tr, y_tr)):
                X_inner_tr = X_tr[inner_tr_idx]
                y_inner_tr = y_tr[inner_tr_idx]

                X_inner_va = X_tr[inner_va_idx]
                y_inner_va = y_tr[inner_va_idx]

                model, train_bce = fit_bce_gam_lbfgs(
                    make_model_fn,
                    X_inner_tr,
                    y_inner_tr,
                    lambda_lin=float(lambda_lin),
                    lambda_smooth=float(lambda_smooth),
                    non_spline_idx=non_spline_idx,
                    spline_block_slices=spline_block_slices,
                    max_iter=lbfgs_max_iter,
                    device=device,
                )

                Xva_t = torch.tensor(X_inner_va, dtype=torch.float32, device=device_t)
                yva_t = torch.tensor(y_inner_va, dtype=torch.float32, device=device_t).view(-1)

                val_bce = _bce_logits_loss(model, Xva_t, yva_t)
                fold_bces.append(float(val_bce))

            mean_cv_bce = float(np.mean(fold_bces))

            rows.append({
                "lambda_lin": float(lambda_lin),
                "lambda_smooth": float(lambda_smooth),
                "cv_bce_mean": mean_cv_bce,
                "cv_bce_folds": fold_bces,
            })

            if (best is None) or (mean_cv_bce < best[2]):
                best = (float(lambda_lin), float(lambda_smooth), mean_cv_bce)

    lambda_lin_star, lambda_smooth_star, best_cv_bce = best
    return lambda_lin_star, lambda_smooth_star, best_cv_bce, rows

## Step 8 — GAM regularization during Smooth Net Benefit training

This step constructs the GAM-specific regularization function used during Smooth Net Benefit training.

The penalty combines the ridge penalty on non-spline coefficients with the second-order smoothness penalty applied within each spline basis block. The regularization strengths selected during BCE model development are retained during subsequent Smooth Net Benefit optimization.

The combined penalty is passed to the generic Smooth Net Benefit trainer so that the structure and smoothness constraints of the BCE-trained GAM remain consistent when the training objective is changed from binary cross-entropy to Smooth Net Benefit.

In [25]:

from nbloss.trainer import nb_anneal_only_with_l2
from nbloss.metrics import average_nb_over_range


def make_gam_penalty_fn(
    *,
    lambda_lin: float,
    lambda_smooth: float,
    non_spline_idx,
    spline_block_slices,
):
    """
    Creates the GAM-specific penalty function used by the generic SNB trainer.

    Important:
    - The trainer already multiplies penalty_fn(model) by l2_lambda.
    - Therefore, when calling the trainer, set l2_lambda=1.0.
    - Bias is not penalized as long as the underlying penalty functions use
      model.linear.weight rather than model.parameters().
    """

    def gam_penalty_fn(model: nn.Module) -> torch.Tensor:
        pen_lin = _linear_l2_penalty_from_idx(
            model,
            non_spline_idx,
        )

        pen_smooth = _smoothness_penalty_from_blocks(
            model,
            spline_block_slices,
        )

        return (
            float(lambda_lin) * pen_lin
            + float(lambda_smooth) * pen_smooth
        )

    return gam_penalty_fn

## Step 9 — Local calibration and Net Benefit evaluation

This step defines the prediction, local calibration, and decision-analytic evaluation functions used after GAM fitting.

Model logits are converted to predicted probabilities using the sigmoid function. Net Benefit is evaluated by averaging across a dense grid of thresholds within the predefined threshold band.

Two post-hoc local calibration methods are implemented as comparators: local temperature scaling and local Platt scaling. For each reference threshold, calibration is estimated using observations from the outer training set whose predicted probabilities lie within a local region around that threshold.

The local interval is expanded when necessary until sufficient positive and negative observations are available. The estimated calibration transformation is then applied to predictions from the independent outer test fold.

Temperature scaling estimates a single scaling parameter for the logits, whereas Platt scaling estimates both a slope and an intercept.

In [21]:



def sigmoid_np(logits_np: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(np.asarray(logits_np, dtype=np.float32).reshape(-1))
    return torch.sigmoid(logits_t).detach().cpu().numpy().astype(np.float64)


@torch.no_grad()
def predict_logits_torch(
    model: nn.Module,
    X_np: np.ndarray,
    *,
    device: str = DEVICE,
) -> np.ndarray:
    device_t = torch.device(device)

    model.eval()

    X_t = torch.tensor(
        X_np,
        dtype=torch.float32,
        device=device_t,
    )

    return (
        model(X_t)
        .detach()
        .cpu()
        .view(-1)
        .numpy()
        .astype(np.float64)
    )


def subset_counts(mask, y_np):
    y_sub = np.asarray(y_np).reshape(-1)[np.asarray(mask, dtype=bool)]

    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))

    return int(len(y_sub)), n_pos, n_neg


def find_local_calibration_range(
    probs_train: np.ndarray,
    y_train: np.ndarray,
    *,
    t_ref: float,
    start_half_width: float = LOCAL_START_HALF_WIDTH,
    expand_step: float = LOCAL_EXPAND_STEP,
    min_pos: int = LOCAL_MIN_POS,
    min_neg: int = LOCAL_MIN_NEG,
) -> dict:
    probs_train = np.asarray(probs_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)

    half_width = float(start_half_width)
    n_expand_steps = 0

    while True:
        low = max(0.0, float(t_ref) - half_width)
        high = min(1.0, float(t_ref) + half_width)

        mask = (probs_train >= low) & (probs_train <= high)
        n, n_pos, n_neg = subset_counts(mask, y_train)

        met_minimum = (n_pos >= int(min_pos)) and (n_neg >= int(min_neg))
        used_full_range = (low <= 0.0) and (high >= 1.0)

        if met_minimum or used_full_range:
            return {
                "low": float(low),
                "high": float(high),
                "half_width": float(half_width),
                "range_width": float(high - low),
                "n_expand_steps": int(n_expand_steps),
                "n": int(n),
                "n_pos": int(n_pos),
                "n_neg": int(n_neg),
                "met_minimum": bool(met_minimum),
                "used_full_range": bool(used_full_range),
                "mask": mask,
            }

        half_width += float(expand_step)
        n_expand_steps += 1


def fit_temperature_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = TEMP_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    log_temperature = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        loss = bce(logits / temperature, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite temperature loss: {loss.detach().item()}"
            )

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        nll_after = float(
            bce(logits / temperature, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "temperature": float(temperature.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_temperature_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = logits_out[mask] / float(temperature)

    return logits_out, mask


def fit_platt_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = PLATT_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    slope = torch.ones(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    intercept = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [slope, intercept],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        calibrated_logits = slope * logits + intercept
        loss = bce(calibrated_logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite Platt loss: {loss.detach().item()}"
            )

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        calibrated_logits = slope * logits + intercept

        nll_after = float(
            bce(calibrated_logits, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "platt_slope": float(slope.detach().cpu().item()),
        "platt_intercept": float(intercept.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_platt_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    slope: float,
    intercept: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = (
        float(slope) * logits_out[mask]
        + float(intercept)
    )

    return logits_out, mask


def evaluate_nb_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    logits_t = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1)
    )

    y_t = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1)
    )

    return float(
        average_nb_over_range(
            logits_t,
            y_t,
            thresh_min=float(thresh_min),
            thresh_max=float(thresh_max),
            num_points=int(num_points),
            input_is_logit=True,
            method="mean",
        )
    )

## Step 10 — Main GAM benchmark

This step executes the complete GAM benchmark across the predefined OpenML datasets.

For each eligible dataset, five outer train/test rotations are evaluated. Within each outer training fold, GAM preprocessing is fitted and the two GAM regularization parameters are selected by inner cross-validation.

The resulting BCE-trained GAM serves as the baseline model and as the initialization for Smooth Net Benefit training.

For each reference decision threshold, the following approaches are evaluated on the same independent outer test fold:

- BCE-trained GAM;
- BCE-trained GAM with local temperature scaling;
- BCE-trained GAM with local Platt scaling;
- Smooth Net Benefit-trained GAM;
- Smooth Net Benefit-trained GAM with local temperature scaling;
- Smooth Net Benefit-trained GAM with local Platt scaling.

Smooth Net Benefit training is performed separately for each decision-relevant threshold band while retaining the regularization parameters selected for the BCE model.

Net Benefit is evaluated over the corresponding threshold interval. The benchmark stores dataset characteristics, fold information, selected regularization parameters, calibration diagnostics, test-set Net Benefit, and differences relative to the BCE baseline.

In [26]:
from copy import deepcopy

results = []


def add_result_row_gam(
    *,
    openml_id: int,
    dataset_name: str,
    run_id: int,
    threshold_name: str,
    model_name: str,
    t_ref: float,
    t_min: float,
    t_max: float,
    prevalence_train: float,
    prevalence_test: float,
    lambda_lin: float,
    lambda_smooth: float,
    cv_bce: float,
    train_bce: float,
    test_nb: float,
    delta_vs_bce: float,
    phi_dim: int,
    local_info: dict | None = None,
    calibration_info: dict | None = None,
):
    local_info = local_info or {}
    calibration_info = calibration_info or {}

    results.append(
        {
            "openml_id": int(openml_id),
            "dataset": str(dataset_name),
            "run_id": int(run_id),
            "threshold_name": str(threshold_name),
            "model": str(model_name),

            "prevalence_train": float(prevalence_train),
            "prevalence_test": float(prevalence_test),

            "t_ref": float(t_ref),
            "t_min": float(t_min),
            "t_max": float(t_max),

            "lambda_lin": float(lambda_lin),
            "lambda_smooth": float(lambda_smooth),
            "cv_bce": float(cv_bce),
            "train_bce": float(train_bce),

            "test_nb": float(test_nb),
            "delta_vs_bce": float(delta_vs_bce),

            "local_low": local_info.get("low", np.nan),
            "local_high": local_info.get("high", np.nan),
            "local_half_width": local_info.get("half_width", np.nan),
            "local_range_width": local_info.get("range_width", np.nan),
            "local_expand_steps": local_info.get("n_expand_steps", np.nan),
            "local_train_n": local_info.get("n", np.nan),
            "local_train_pos": local_info.get("n_pos", np.nan),
            "local_train_neg": local_info.get("n_neg", np.nan),
            "local_met_minimum": local_info.get("met_minimum", np.nan),
            "local_used_full_range": local_info.get("used_full_range", np.nan),

            "temperature": calibration_info.get("temperature", np.nan),
            "platt_slope": calibration_info.get("platt_slope", np.nan),
            "platt_intercept": calibration_info.get("platt_intercept", np.nan),
            "train_local_nll_before": calibration_info.get("nll_before", np.nan),
            "train_local_nll_after": calibration_info.get("nll_after", np.nan),

            "phi_dim": int(phi_dim),
        }
    )


for did in OPENML_DATASET_IDS:
    print(f"\n==================== OpenML dataset {did} ====================")

    try:
        ds, X_df, y, meta = load_openml_dataset(did)
    except Exception as e:
        print(f"[SKIP] Could not load dataset {did}: {repr(e)}")
        continue

    openml_id = int(meta["openml_id"])
    dataset_name = str(meta["name"])

    if int(meta["n_rows"]) < int(MIN_ROWS):
        print(
            f"[SKIP] {dataset_name} ({openml_id}) has only "
            f"{meta['n_rows']} rows < MIN_ROWS={MIN_ROWS}"
        )
        continue

    y = np.asarray(y, dtype=np.float32).reshape(-1)

    if len(np.unique(y)) != 2:
        print(f"[SKIP] {dataset_name} ({openml_id}) is not binary after loading.")
        continue

    folds = make_5fold_indices(y, seed=SEED_GLOBAL)

    for run_id in range(5):
        print(f"\n---------- {dataset_name} | run {run_id} ----------")

        train_idx, test_idx = indices_for_run_train_test(folds, run_id)

        X_tr_df = X_df.iloc[train_idx].copy()
        X_te_df = X_df.iloc[test_idx].copy()

        y_tr = y[train_idx].astype(np.float32).reshape(-1)
        y_te = y[test_idx].astype(np.float32).reshape(-1)

        prevalence_train = float(y_tr.mean())
        prevalence_test = float(y_te.mean())

        try:
            prep = fit_train_preprocessor_spline_and_transform(
                X_tr_df,
                X_te_df,
                spline_n_knots=10,
                spline_degree=3,
            )
        except Exception as e:
            print(
                f"[SKIP RUN] preprocessing failed for "
                f"{dataset_name} | run {run_id}: {repr(e)}"
            )
            continue

        Phi_tr = np.asarray(prep["Phi_tr"], dtype=np.float32)
        Phi_te = np.asarray(prep["Phi_te"], dtype=np.float32)

        non_spline_idx = prep["non_spline_idx"]
        spline_block_slices = prep["spline_block_slices"]

        d_in_phi = int(Phi_tr.shape[1])

        def make_gam_model():
            return TorchGAM(d_in_phi)

        try:
            lambda_lin_star, lambda_smooth_star, best_cv_bce, cv_rows = (
                select_gam_penalties_by_cv_bce(
                    LAMBDA_LIN_GRID,
                    LAMBDA_SMOOTH_GRID,
                    make_model_fn=make_gam_model,
                    X_tr=Phi_tr,
                    y_tr=y_tr,
                    non_spline_idx=non_spline_idx,
                    spline_block_slices=spline_block_slices,
                    device=DEVICE,
                    n_splits=5,
                    lbfgs_max_iter=500,
                    seed=SEED_GLOBAL + run_id,
                )
            )

            print(
                f"[{dataset_name} | run {run_id}] "
                f"prev_train={prevalence_train:.4f} | "
                f"lambda_lin={lambda_lin_star:g} | "
                f"lambda_smooth={lambda_smooth_star:g} | "
                f"best inner-CV BCE={best_cv_bce:.6f}"
            )

            train_dl = make_loader(
                Phi_tr,
                y_tr,
                batch=BATCH,
                shuffle=True,
                seed=SEED_GLOBAL + run_id,
            )

            bce_model, train_bce = fit_bce_gam_lbfgs(
                make_gam_model,
                Phi_tr,
                y_tr,
                lambda_lin=float(lambda_lin_star),
                lambda_smooth=float(lambda_smooth_star),
                non_spline_idx=non_spline_idx,
                spline_block_slices=spline_block_slices,
                max_iter=500,
                device=DEVICE,
            )

            logits_bce_train = predict_logits_torch(
                bce_model,
                Phi_tr,
                device=DEVICE,
            )

            logits_bce_test = predict_logits_torch(
                bce_model,
                Phi_te,
                device=DEVICE,
            )

            probs_bce_train = sigmoid_np(logits_bce_train)
            probs_bce_test = sigmoid_np(logits_bce_test)

            threshold_specs = threshold_specs_from_prevalence(
                prevalence_train,
                mid_prev_low=MID_PREV_LOW,
                mid_prev_high=MID_PREV_HIGH,
            )

            for spec in threshold_specs:
                threshold_name = spec["threshold_name"]
                t_ref = float(spec["threshold"])
                t_min, t_max = band_from_t_ref(
                    t_ref,
                    half_width=BAND_HALF_WIDTH,
                )

                print(
                    f"\n[{dataset_name} | run {run_id}] "
                    f"=== THRESHOLD: {threshold_name} "
                    f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
                )

                nb_bce = evaluate_nb_from_logits_np(
                    logits_bce_test,
                    y_te,
                    thresh_min=t_min,
                    thresh_max=t_max,
                    num_points=TEST_RANGE_POINTS,
                )

                local_bce = find_local_calibration_range(
                    probs_bce_train,
                    y_tr,
                    t_ref=float(t_ref),
                    start_half_width=LOCAL_START_HALF_WIDTH,
                    expand_step=LOCAL_EXPAND_STEP,
                    min_pos=LOCAL_MIN_POS,
                    min_neg=LOCAL_MIN_NEG,
                )

                logits_bce_train_local = logits_bce_train[local_bce["mask"]]
                ytr_bce_local = y_tr[local_bce["mask"]]

                temp_bce_fit = fit_temperature_from_logits_np(
                    logits_bce_train_local,
                    ytr_bce_local,
                    max_iter=TEMP_MAX_ITER,
                    device=DEVICE,
                )

                platt_bce_fit = fit_platt_from_logits_np(
                    logits_bce_train_local,
                    ytr_bce_local,
                    max_iter=PLATT_MAX_ITER,
                    device=DEVICE,
                )

                logits_bce_temp_test, _ = apply_local_temperature_to_logits(
                    logits_bce_test,
                    probs_bce_test,
                    low=local_bce["low"],
                    high=local_bce["high"],
                    temperature=temp_bce_fit["temperature"],
                )

                logits_bce_platt_test, _ = apply_local_platt_to_logits(
                    logits_bce_test,
                    probs_bce_test,
                    low=local_bce["low"],
                    high=local_bce["high"],
                    slope=platt_bce_fit["platt_slope"],
                    intercept=platt_bce_fit["platt_intercept"],
                )

                nb_bce_temp = evaluate_nb_from_logits_np(
                    logits_bce_temp_test,
                    y_te,
                    thresh_min=t_min,
                    thresh_max=t_max,
                    num_points=TEST_RANGE_POINTS,
                )

                nb_bce_platt = evaluate_nb_from_logits_np(
                    logits_bce_platt_test,
                    y_te,
                    thresh_min=t_min,
                    thresh_max=t_max,
                    num_points=TEST_RANGE_POINTS,
                )

                snb_start = make_gam_model().to(DEVICE)
                snb_start.load_state_dict(
                    {
                        k: v.detach().cpu().clone()
                        for k, v in bce_model.state_dict().items()
                    }
                )

                gam_penalty_fn = make_gam_penalty_fn(
                    lambda_lin=float(lambda_lin_star),
                    lambda_smooth=float(lambda_smooth_star),
                    non_spline_idx=non_spline_idx,
                    spline_block_slices=spline_block_slices,
                )

                snb_model = nb_anneal_only_with_l2(
                    model=snb_start,
                    train_dl=train_dl,
                    thresh_min=float(t_min),
                    thresh_max=float(t_max),
                    num_points_train=int(TRAIN_RANGE_POINTS),
                    inverse_temps=tuple(INVERSE_TEMPS),
                    epochs_per_temp=int(EPOCHS_PER_TEMP),
                    patience_hard=int(PATIENCE_HARD),
                    hard_range_num_points=int(TEST_RANGE_POINTS),
                    lr_adam=float(LR_ADAMW),
                    l2_lambda=1.0,
                    penalty_fn=gam_penalty_fn,
                    device=DEVICE,
                    seed=int(MODEL_SEED) + 1000 * int(run_id) + 17,
                    log_every=20,
                )

                logits_snb_test = predict_logits_torch(
                    snb_model,
                    Phi_te,
                    device=DEVICE,
                )

                nb_snb = evaluate_nb_from_logits_np(
                    logits_snb_test,
                    y_te,
                    thresh_min=t_min,
                    thresh_max=t_max,
                    num_points=TEST_RANGE_POINTS,
                )

                add_result_row_gam(
                    openml_id=openml_id,
                    dataset_name=dataset_name,
                    run_id=run_id,
                    threshold_name=threshold_name,
                    model_name="bce_gam",
                    t_ref=t_ref,
                    t_min=t_min,
                    t_max=t_max,
                    prevalence_train=prevalence_train,
                    prevalence_test=prevalence_test,
                    lambda_lin=lambda_lin_star,
                    lambda_smooth=lambda_smooth_star,
                    cv_bce=best_cv_bce,
                    train_bce=train_bce,
                    test_nb=nb_bce,
                    delta_vs_bce=0.0,
                    phi_dim=d_in_phi,
                )

                add_result_row_gam(
                    openml_id=openml_id,
                    dataset_name=dataset_name,
                    run_id=run_id,
                    threshold_name=threshold_name,
                    model_name="bce_gam_local_temperature",
                    t_ref=t_ref,
                    t_min=t_min,
                    t_max=t_max,
                    prevalence_train=prevalence_train,
                    prevalence_test=prevalence_test,
                    lambda_lin=lambda_lin_star,
                    lambda_smooth=lambda_smooth_star,
                    cv_bce=best_cv_bce,
                    train_bce=train_bce,
                    test_nb=nb_bce_temp,
                    delta_vs_bce=nb_bce_temp - nb_bce,
                    phi_dim=d_in_phi,
                    local_info=local_bce,
                    calibration_info=temp_bce_fit,
                )

                add_result_row_gam(
                    openml_id=openml_id,
                    dataset_name=dataset_name,
                    run_id=run_id,
                    threshold_name=threshold_name,
                    model_name="bce_gam_local_platt",
                    t_ref=t_ref,
                    t_min=t_min,
                    t_max=t_max,
                    prevalence_train=prevalence_train,
                    prevalence_test=prevalence_test,
                    lambda_lin=lambda_lin_star,
                    lambda_smooth=lambda_smooth_star,
                    cv_bce=best_cv_bce,
                    train_bce=train_bce,
                    test_nb=nb_bce_platt,
                    delta_vs_bce=nb_bce_platt - nb_bce,
                    phi_dim=d_in_phi,
                    local_info=local_bce,
                    calibration_info=platt_bce_fit,
                )

                add_result_row_gam(
                    openml_id=openml_id,
                    dataset_name=dataset_name,
                    run_id=run_id,
                    threshold_name=threshold_name,
                    model_name="snb_gam",
                    t_ref=t_ref,
                    t_min=t_min,
                    t_max=t_max,
                    prevalence_train=prevalence_train,
                    prevalence_test=prevalence_test,
                    lambda_lin=lambda_lin_star,
                    lambda_smooth=lambda_smooth_star,
                    cv_bce=best_cv_bce,
                    train_bce=train_bce,
                    test_nb=nb_snb,
                    delta_vs_bce=nb_snb - nb_bce,
                    phi_dim=d_in_phi,
                )

                print(
                    f"[TEST][{threshold_name}] "
                    f"BCE={nb_bce:.6f} | "
                    f"BCE-temp={nb_bce_temp:.6f} "
                    f"({nb_bce_temp - nb_bce:+.6f}) | "
                    f"BCE-Platt={nb_bce_platt:.6f} "
                    f"({nb_bce_platt - nb_bce:+.6f}) | "
                    f"SNB={nb_snb:.6f} "
                    f"({nb_snb - nb_bce:+.6f})"
                )

        except Exception as e:
            print(
                f"[SKIP RUN] failed for {dataset_name} "
                f"({openml_id}) | run {run_id}: {repr(e)}"
            )
            continue

        if DEVICE == "cuda":
            torch.cuda.empty_cache()


results_df = pd.DataFrame(results)

print("\nFinished.")
print("results_df shape:", results_df.shape)
display(results_df.head())


==================== OpenML dataset 1120 ====================

---------- MagicTelescope | run 0 ----------
[MagicTelescope | run 0] prev_train=0.3517 | lambda_lin=0.01 | lambda_smooth=0.0001 | best inner-CV BCE=0.369277

[MagicTelescope | run 0] === THRESHOLD: prev (t_ref=0.3517, [0.3267, 0.3767]) ===
[SNB start] initial train hard NB range = 0.228802
[SNB inverse_temp=1] epoch 020 | train_loss=-0.206279 | train_hard_nb=0.225747
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=1 no improvement; keep global train hard NB: 0.228802
[SNB inverse_temp=4] epoch 020 | train_loss=-0.223492 | train_hard_nb=0.231472
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=4 improved global train hard NB: 0.228802 → 0.232128
[SNB inverse_temp=10] epoch 020 | train_loss=-0.228838 | train_hard_nb=0.233867
[SNB inverse_temp=10] epoch 040 | train_loss=-0.229285 | train_hard_nb=0.233895
[SNB inverse_temp=10] epoch 060 | train_loss=-0.228777 | train_hard_nb=0.234011
>>> Early s

KeyboardInterrupt: 